# Lab 2 SOLUTIONS: RAG Evaluation with DeepEval
## CCI Session 6

In [ ]:
!pip install -q llama-parse langchain langchain-openai langchain-community langchain-chroma chromadb deepeval

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['LLAMA_CLOUD_API_KEY'] = userdata.get('LLAMA_CLOUD_API_KEY')
from google.colab import files
files.upload()
PDF_PATH = '/content/WT.pdf'

## Cell 1 — Test set

In [ ]:
test_cases = [
    {
        'input': 'What chemotherapy regimen is recommended for Stage IV Wilms tumor?',
        'expected_output': 'Stage IV Wilms tumor with favorable histology is treated with regimen DD4A or M (vincristine, dactinomycin, doxorubicin) plus radiation to the abdomen and lungs as needed.',
        'retrieval_context': ['Stage IV favorable histology Wilms tumor is treated with three-drug chemotherapy (vincristine, dactinomycin, doxorubicin) plus radiation therapy.']
    },
    {
        'input': 'What is the standard treatment for Stage I favorable histology Wilms tumor in children under 2?',
        'expected_output': 'Surgery alone (nephrectomy) may be considered for very low risk Stage I FH tumors <550g in children <2 years; otherwise EE-4A regimen (vincristine + dactinomycin) for 18 weeks.',
        'retrieval_context': ['Stage I FH WT in children younger than 2 years with tumor weight <550 grams may be treated by nephrectomy alone.']
    },
    {
        'input': 'What are the dose-limiting toxicities of vincristine?',
        'expected_output': 'Peripheral neuropathy (sensory and motor), constipation/ileus, jaw pain, and SIADH are common dose-limiting toxicities.',
        'retrieval_context': ['Vincristine is associated with peripheral neuropathy, constipation, and jaw pain.']
    },
    {
        'input': 'How is diffuse anaplastic histology Wilms tumor treated?',
        'expected_output': 'Diffuse anaplastic Wilms tumor requires intensified regimens such as Regimen UH-1 or UH-2 with cyclophosphamide, carboplatin, etoposide, doxorubicin, vincristine plus radiation.',
        'retrieval_context': ['Diffuse anaplastic histology requires intensified multi-agent chemotherapy and radiation.']
    },
    {
        'input': 'When is flank radiation indicated in Wilms tumor?',
        'expected_output': 'Flank radiation is indicated for Stage III tumors, residual disease after surgery, positive margins, lymph node involvement, and tumor spill.',
        'retrieval_context': ['Stage III Wilms tumor with positive nodes, tumor spill, or residual disease requires flank radiation.']
    },
]

In [ ]:
from llama_parse import LlamaParse
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

raw_docs = LlamaParse(api_key=os.environ['LLAMA_CLOUD_API_KEY'], result_type='markdown').load_data(PDF_PATH)
lc_docs = [Document(page_content=d.text, metadata={'source': 'WT.pdf'}) for d in raw_docs]

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ('system', 'Answer ONLY from context.\n\n{context}'),
    ('human', '{input}')
])

## Cell 2 — BAD RAG

In [ ]:
bad_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
bad_chunks = bad_splitter.split_documents(lc_docs)
bad_vs = Chroma.from_documents(bad_chunks, embeddings, collection_name='bad_rag')
bad_retriever = bad_vs.as_retriever(search_kwargs={'k': 4})
bad_chain = create_retrieval_chain(bad_retriever, create_stuff_documents_chain(llm, prompt))

## Cell 3 — GOOD RAG

In [ ]:
good_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
good_chunks = good_splitter.split_documents(lc_docs)
good_vs = Chroma.from_documents(good_chunks, embeddings, collection_name='good_rag')
good_retriever = good_vs.as_retriever(search_kwargs={'k': 4})
good_chain = create_retrieval_chain(good_retriever, create_stuff_documents_chain(llm, prompt))

## Cell 4 — Metrics

In [ ]:
from deepeval.metrics import (FaithfulnessMetric, AnswerRelevancyMetric,
    ContextualRelevancyMetric, ContextualRecallMetric)
from deepeval.test_case import LLMTestCase

MODEL = 'gpt-4o-mini'
metrics = [
    FaithfulnessMetric(threshold=0.7, model=MODEL),
    AnswerRelevancyMetric(threshold=0.7, model=MODEL),
    ContextualRelevancyMetric(threshold=0.7, model=MODEL),
    ContextualRecallMetric(threshold=0.7, model=MODEL),
]

## Cell 5 — Evaluate

In [ ]:
def evaluate(chain):
    rows = []
    for tc in test_cases:
        res = chain.invoke({'input': tc['input']})
        case = LLMTestCase(
            input=tc['input'],
            actual_output=res['answer'],
            expected_output=tc['expected_output'],
            retrieval_context=[d.page_content for d in res['context']],
        )
        scores = {'question': tc['input'][:60]}
        for m in metrics:
            m.measure(case)
            scores[m.__class__.__name__] = round(m.score, 3)
        rows.append(scores)
    return rows

bad_results = evaluate(bad_chain)
good_results = evaluate(good_chain)

## Cell 6 — Compare

In [ ]:
import pandas as pd
df_bad = pd.DataFrame(bad_results)
df_good = pd.DataFrame(good_results)
print('--- BAD ---'); print(df_bad)
print('\n--- GOOD ---'); print(df_good)
summary = pd.DataFrame({
    'BAD': df_bad.drop(columns=['question']).mean(),
    'GOOD': df_good.drop(columns=['question']).mean(),
})
print('\nMean comparison:'); print(summary)

## Cell 7 — Diagnose

In [ ]:
worst_idx = df_bad['FaithfulnessMetric'].idxmin()
tc = test_cases[worst_idx]
res = bad_chain.invoke({'input': tc['input']})
print('Worst BAD case:', tc['input'])
print('Answer:', res['answer'])
for i, d in enumerate(res['context']):
    print(f'\nChunk {i}:', d.page_content)

## Stretch — Custom G-Eval

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

clinical_safety = GEval(
    name='ClinicalSafety',
    criteria='The output must not contain any unsafe drug-dosing claims and must defer to oncology guidelines when uncertain.',
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.7, model='gpt-4o-mini',
)